In [31]:
# Cell 1 - Imports
import requests
import pandas as pd
import time


In [32]:
# Cell 2 — Configuration (only this cell needs to be updated)
# BASE_URL = "https://findata-qa.worldbank.org:9047"
# TOKEN    = "3/Cw8MfRR5CgJxP9cfVlot3cd95gGBfZvwEa7jp3tBOU9UWfFCJ8e6k6KAhF0w=="
# SCHEMA   = ["publishFINDATAREP", "IFCTRE", "Application", "Murex"]

BASE_URL = "https://findata-dev.worldbank.org:9047/"   # DR4
TOKEN    = "tgpnYKvYSueh5po/dtAS8niyCKKL/KpzDQ6jfAttdO40edAtbTvpYMO+2sE08g==" 
SCHEMA   = ["publishFINDATAREP1", "IFCTRE", "Application", "Murex"]

HEADERS = {
    "Authorization": f"Bearer {TOKEN}",
    "Content-Type": "application/json"
}

# ── Date parameter — update this to change the filter ──
BUSINESS_DATE = "2026-08-14"

# ── All your queries ──
QUERIES = {
    "perf_attribution"      : 'SELECT * FROM "Perf_Attribution"',
    "perf_benchmark"        : 'SELECT * FROM "Perf_Benchmark_Weights"',
    "perf_vs_funding"       : 'SELECT * FROM "Perf_Vs_Funding_Snap"',
    "plvar_snapshot"        : 'SELECT * FROM "Plvar_Snapshot"',
    "portfolio_details"     : 'SELECT * FROM "Portfolio_Details"',
    "securities_detail"     : 'SELECT * FROM "Securities_Detail"',
    "trade_mktval_detail"   : 'SELECT * FROM "Trade_Mktval_Detail"',
    "security_prices_detail": 'SELECT * FROM "Security_Prices_Detail"',
    "inv_pnl_summary"       : f'SELECT * FROM "Inv_Pnl_Summary" WHERE "Business_Date" = \'{BUSINESS_DATE}\'',
    "inv_security_position" : 'SELECT * FROM "Inv_Security_Position"',
}

# ── Only change this line to switch query ──
ACTIVE_QUERY =   "inv_pnl_summary" 
SQL_QUERY    = QUERIES[ACTIVE_QUERY]


In [33]:
# Cell 3 — Display function
def display_df(df):
    return df.style.set_properties(**{
        'white-space': 'nowrap',
        'font-size': '11px',
        'text-align': 'left',
        'border': '1px solid lightgrey',
        'color': 'black'
    }).set_table_styles([
        {
            'selector': 'thead th',
            'props': [
                ('white-space', 'nowrap'),
                ('font-size', '11px'),
                ('font-weight', 'bold'),
                ('background-color', '#4472C4'),
                ('color', 'white'),
                ('text-align', 'center'),
                ('border', '1px solid lightgrey')
            ]
        },
        {
            'selector': 'tbody tr:nth-child(even)',
            'props': [('background-color', '#f2f2f2'), ('color', 'black')]
        },
        {
            'selector': 'tbody tr:nth-child(odd)',
            'props': [('background-color', 'white'), ('color', 'black')]
        },
        {
            'selector': 'tbody tr:hover',
            'props': [('background-color', '#d6e4f0'), ('color', 'black')]
        }
    ])


In [34]:
# Cell 4 — Submit the SQL job
response = requests.post(
    f"{BASE_URL}/api/v3/sql",
    headers=HEADERS,
    json={
        "sql": SQL_QUERY,
        "context": SCHEMA
    },
    verify=True,
    timeout=30
)
response.raise_for_status()
job_id = response.json().get("id")
print(f"Job submitted successfully.")
print(f"Job ID: {job_id}")


Job submitted successfully.
Job ID: 157e4aff-9033-2a31-4fed-379da7724800


In [35]:
# Cell 5 — Poll until job completes
url = f"{BASE_URL}/api/v3/job/{job_id}"
for attempt in range(30):
    response = requests.get(url, headers=HEADERS, verify=True)
    response.raise_for_status()
    status = response.json().get("jobState")
    print(f"Attempt {attempt + 1}: {status}")
    if status == "COMPLETED":
        print("Job completed!")
        break
    elif status in ("FAILED", "CANCELED"):
        print(f"Job failed: {response.json().get('errorMessage')}")
        break
    time.sleep(2)


Attempt 1: COMPLETED
Job completed!


In [37]:
# Cell 6 — Fetch results and build DataFrame

# ── Column selection per query ──
COLUMNS = {
    "perf_attribution"      : [],
    "perf_benchmark"        : [],
    "perf_vs_funding"       : [],
    "plvar_snapshot"        : [],
    "portfolio_details"     : [],
    "securities_detail"     : [],
    "trade_mktval_detail"   : [],
    "security_prices_detail": [],
    "inv_pnl_summary"       : [
        "Business_Date", "MX_L3_PORTFO", "MX_L4_PORTFO", "Currency", "MX_CD_ELIG", "Trade_Typology",
        "Instrument", "Contract_Origin", "Trade_ID",
        "Country", "CTP_Code", "MX_DEAL_TYPE", "DTD_pnl_usd", "MTD_Pnl_USD",
        "YTD_pnl_USD", "LTD_pnl_usd", "Cash", "Cash_USD", "Market_Value_USD"
    ],
    "inv_security_position" : [],
}

# ── Rename mapping per query ──
RENAME = {
    "inv_pnl_summary" : {
        "Business_Date"   : "Business_Date",
        "MX_L3_PORTFO"   : "L3_PORTFO",
        "MX_L4_PORTFO"   : "L4_PORTFO",
        "Currency"        : "CURRENCY",
        "MX_CD_ELIG"     : "CD_ELIG",
        "Trade_Typology"  : "TYPOLOGY",
        "Instrument"      : "INSTRUMEN",
        "Contract_Origin" : "CNT_ORG",
        "Trade_ID"        : "NB",
        "Country"         : "COUNTRY",
        "CTP_Code"        : "CTP",
        "MX_DEAL_TYPE"   : "DEAL_TYPE",
        "DTD_pnl_usd"    : "DTD_PL_US",
        "MTD_Pnl_USD"    : "MTD_PL_US",
        "YTD_pnl_USD"    : "YTD_PL_US",
        "LTD_pnl_usd"    : "LTD_PL_US",
        "Cash"            : "CASH",
        "Cash_USD"        : "CASH_US",
        "Market_Value_USD": "MV_US",
    },
}

# ── Fetch first page ──
response = requests.get(
    f"{BASE_URL}/api/v3/job/{job_id}/results?offset=0&limit=500",
    headers=HEADERS,
    verify=True
)
response.raise_for_status()
data = response.json()

columns  = [col["name"] for col in data.get("schema", [])]
all_rows = data.get("rows", [])

# ── Paginate if more than 500 rows ──
offset = 500
while True:
    r = requests.get(
        f"{BASE_URL}/api/v3/job/{job_id}/results?offset={offset}&limit=500",
        headers=HEADERS,
        verify=True
    )
    batch = r.json().get("rows", [])
    if not batch:
        break
    all_rows.extend(batch)
    offset += len(batch)
    print(f"Fetched {offset} rows so far...")

df = pd.DataFrame(all_rows, columns=columns)

# ── Inspect all available columns before filtering ──
print(f"\nAll available columns ({len(df.columns)}):")
for col in df.columns:
    print(f"  - {col}")

# ── Apply column selection ──
selected_cols = COLUMNS.get(ACTIVE_QUERY, [])
if selected_cols:
    missing = [c for c in selected_cols if c not in df.columns]
    if missing:
        print(f"\n⚠️  Columns not found and skipped: {missing}")
    df = df[[c for c in selected_cols if c in df.columns]]

# ── Apply column renaming ──
rename_map = RENAME.get(ACTIVE_QUERY, {})
if rename_map:
    df = df.rename(columns=rename_map)

print(f"\nActive Query  : {ACTIVE_QUERY}")
print(f"Total Rows    : {len(df)}")
print(f"Total Columns : {len(df.columns)}")
print(f"Columns       : {list(df.columns)}")


Fetched 1000 rows so far...
Fetched 1500 rows so far...
Fetched 1966 rows so far...

All available columns (49):
  - Business_Date
  - MX_REF_DATA
  - MX_REF_JOB
  - Cash
  - Cash_USD
  - MX_CD_ELIG
  - MX_CD_INC_PTF
  - MX_CD_INC_USD
  - Contract_Origin
  - Root_Contract_ID
  - Country
  - CTP_Code
  - Currency
  - MX_DATA_DATE
  - Data_Label
  - MX_DEAL_TYPE
  - DTD_pnl
  - DTD_pnl_usd
  - Data_Generation_Date
  - Data_Generation_Desk
  - Data_Generation_Group
  - Data_Generation_Time
  - Data_Generation_User
  - Instrument
  - Level1_sector
  - Level2_sector
  - MX_L3_PORTFO
  - MX_L4_PORTFO
  - Package_ID
  - LTD_pnl
  - LTD_pnl_usd
  - MTD_Pnl
  - MTD_Pnl_USD
  - Market_Value
  - Market_Value_USD
  - Trade_ID
  - Book_ID
  - MX_PTF_CCY
  - Trade_Type_Desc
  - Tranche_ID
  - Trade_Typology
  - YTD_pnl
  - YTD_pnl_USD
  - UUID
  - FeedName
  - ProcessedTimeStamp
  - AccountingDate
  - BatchID
  - LastProcessedBatchId

Active Query  : inv_pnl_summary
Total Rows    : 1966
Total Column

In [38]:
mask = (
    df["L3_PORTFO"].isin(["LAM", "LAM_LIQUIDTY", "LAM_STRATEGIC"]) &
    (df["TYPOLOGY"] == "Investment Bond")
)

df_filtered = df[mask].reset_index(drop=True)

print(f"Rows before filter : {len(df)}")
print(f"Rows after filter  : {len(df_filtered)}")


Rows before filter : 1966
Rows after filter  : 722


In [39]:
display_df(df_filtered.head(10))   # first 10 rows

,Business_Date,L3_PORTFO,L4_PORTFO,CURRENCY,CD_ELIG,TYPOLOGY,INSTRUMEN,CNT_ORG,NB,COUNTRY,CTP,DEAL_TYPE,DTD_PL_US,MTD_PL_US,YTD_PL_US,LTD_PL_US,CASH,CASH_US,MV_US
0,2026-08-14,LAM_STRATEGIC,LAM_STG_GRE,KRW,N,Investment Bond,INDKOR 2.9 080128 2601,1724233,1724828,KR,HYSEKR,Asset,87822.680000,87822.680000,87822.680000,87822.680000,-13901073.535509,-13901073.535509,13988896.212288
1,2026-08-14,LAM,NW_LAM_REPO,USD,Y,Investment Bond,SHINLN ECD 0 09/11/26,1724234,1724829,KR,SALBLO,Asset,-19925.880000,-19925.880000,-19925.880000,-19925.880000,-49500743.748500,-49500743.748500,49480817.866861
2,2026-08-14,LAM_STRATEGIC,LAM_STG_CB,USD,N,Investment Bond,BNS 4.299 200328 S REGS,1724235,1724830,CA,DEUTFR,Asset,2993607.030000,2993607.030000,2993607.030000,2993607.030000,-47850500.000000,-47850500.000000,50844107.027713
3,2026-08-14,LAM_STRATEGIC,LAM_STG_GRE,KRW,N,Investment Bond,KDB 4.31 171026 0300,1724236,1724831,KR,HYSEKR,Asset,296004.030000,296004.030000,296004.030000,296004.030000,-13894439.522720,-13894439.522720,14190443.548244
4,2026-08-14,LAM_STRATEGIC,LAM_STG_SP_INT,GBP,N,Investment Bond,TWIN 4.518200 12/12/2055 A 1,1724238,1724833,GB,MLILLO,Asset,-495.250000,-495.250000,-495.250000,-495.250000,-4991572.445969,-4991572.445969,4991077.200452
5,2026-08-14,LAM_STRATEGIC,LAM_STG_SP_USA,USD,N,Investment Bond,WHEELS 4.300000 04/18/2039 A2A 1A,1724239,1724834,US,BNPAUS,Asset,164414.970000,164414.970000,164414.970000,164414.970000,-13310762.972730,-13310762.972730,13475177.939185
6,2026-08-14,LAM_STRATEGIC,LAM_STG_GRE,KRW,N,Investment Bond,KDB 3.75 151227 0106,1724244,1724839,KR,HYSEKR,Asset,491976.680000,491976.680000,491976.680000,491976.680000,-84711082.955085,-84711082.955085,85203059.635858
7,2026-08-14,LAM_STRATEGIC,LAM_STG_SP_USA,USD,N,Investment Bond,XROAD 5.900000 08/20/2030 A2 A,1724245,1724840,US,MLSCUS,Asset,1119599.140000,1119599.140000,1119599.140000,1119599.140000,-2204949.702455,-2204949.702455,3324548.840894
8,2026-08-14,LAM_STRATEGIC,LAM_STG_GRE,USD,N,Investment Bond,CPPIBC 4.125 100630 S REGs,1724253,1724848,CA,TDSSNY,Asset,1141339.800000,1141339.800000,1141339.800000,1141339.800000,-25331085.000000,-25331085.000000,26472424.803784
9,2026-08-14,LAM_STRATEGIC,LAM_STG_GRE,KRW,N,Investment Bond,INDKOR 3.8 181227 2606,1724256,1724851,KR,WISHKR,Asset,543141.310000,543141.310000,543141.310000,543141.310000,-91784468.702921,-91784468.702921,92327610.012347


In [40]:
display_df(df_filtered.tail(10))   # last 10 rows

,Business_Date,L3_PORTFO,L4_PORTFO,CURRENCY,CD_ELIG,TYPOLOGY,INSTRUMEN,CNT_ORG,NB,COUNTRY,CTP,DEAL_TYPE,DTD_PL_US,MTD_PL_US,YTD_PL_US,LTD_PL_US,CASH,CASH_US,MV_US
712,2026-08-14,LAM_STRATEGIC,LAM_STG_CB,CHF,N,Investment Bond,PSHYPO 0 171127 S AUG,1725980,1726575,CH,UBSWLO,Asset,27692.520000,27692.520000,27692.520000,27692.520000,-61173996.488470,-61173996.488470,61201689.003768
713,2026-08-14,LAM_STRATEGIC,LAM_STG_SP_INT,AUD,N,Investment Bond,PROGS 5.427500 06/27/2048 A 1,1725983,1726578,AU,CITIGB,Asset,-1370586.610000,-1370586.610000,-1370586.610000,-1370586.610000,652298.803702,652298.803702,-2022885.409657
714,2026-08-14,LAM_STRATEGIC,LAM_STG_CB,CHF,N,Investment Bond,PSHYPO 1.5 210128 S 562,1725984,1726579,CH,RAIFCH,Asset,3248.470000,3248.470000,3248.470000,3248.470000,-6306383.130399,-6306383.130399,6309631.596075
715,2026-08-14,LAM_STRATEGIC,LAM_STG_SP_INT,AUD,N,Investment Bond,METR 5.912500 04/18/2029 A 1,1725987,1726582,AU,NATAME,Asset,8312935.400000,8312935.400000,8312935.400000,8312935.400000,-8339549.922949,-8339549.922949,16652485.322737
716,2026-08-14,LAM_STRATEGIC,LAM_STG_CB,AUD,N,Investment Bond,CM FRN 65 240728 S MTN,1725989,1726584,CA,NATAME,Asset,6335046.800000,6335046.800000,6335046.800000,6335046.800000,-113672991.507010,-113672991.507010,120008038.306400
717,2026-08-14,LAM_STRATEGIC,LAM_STG_SP_INT,AUD,N,Investment Bond,TRTN 5.709900 08/09/2054 A1AU 1,1725990,1726585,AU,NATAME,Asset,12362891.110000,12362891.110000,12362891.110000,12362891.110000,-15889030.319522,-15889030.319522,28251921.430985
718,2026-08-14,LAM,NW_LAM_REPO,AUD,Y,Investment Bond,RABOAU ECD 0 10/26/26,1725991,1726586,NL,RBOSGR,Asset,0.050000,0.050000,0.050000,0.050000,-69127478.378659,-69127478.378659,69127478.428250
719,2026-08-14,LAM,NW_LAM_USTRACK,USD,N,Investment Bond,T 4.125 150629,1725992,1726587,US,FUNBUS,Asset,526553.820000,526553.820000,526553.820000,526553.820000,-202198559.170082,-202198559.170082,202725112.989615
720,2026-08-14,LAM_STRATEGIC,LAM_STG_GRE,USD,N,Investment Bond,T 4 310330,1725993,1726588,US,SBSIUS,Asset,-869768.140000,-869768.140000,-869768.140000,-869768.140000,-101373712.225275,-101373712.225275,100503944.087383
721,2026-08-14,LAM_STRATEGIC,LAM_STG_SP_INT,AUD,N,Investment Bond,SAPP 5.018200 12/12/2067 A1S 1,1725999,1726594,AU,WPACSY,Asset,1318244.880000,1318244.880000,1318244.880000,1318244.880000,-36280479.998700,-36280479.998700,37598724.879243


In [41]:
# Cell — Export DataFrame to Excel with professional formatting
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
import os
from datetime import datetime

# ── Build filename with date ──
date_str  = BUSINESS_DATE.replace("-", "")           # e.g. 20260811
filename  = f"MX_Subaccount_PL_{date_str}.xlsx"

output_dir = r"C:\Users\mkeita9\OneDrive - WBG\1. Work Documents\4. Projects & Initiatives\Murex_Testing_LAM_Cash\08_DR\DR4\01.Inputs\1_Datamart_Extracts\4_Murex_PL_Summary"

os.makedirs(output_dir, exist_ok=True)
filepath  = os.path.join(output_dir, filename)

# ── Save raw DataFrame first ──
df_filtered.to_excel(filepath, index=False, sheet_name="PL_Summary")

# ── Apply professional formatting ──
wb = load_workbook(filepath)
ws = wb["PL_Summary"]

# Header style
header_font    = Font(name="Calibri", bold=True, color="FFFFFF", size=11)
header_fill    = PatternFill(fill_type="solid", fgColor="003366")   # WBG dark blue
header_align   = Alignment(horizontal="center", vertical="center", wrap_text=True)

# Body style
body_font      = Font(name="Calibri", size=10)
body_align_l   = Alignment(horizontal="left",   vertical="center")
body_align_r   = Alignment(horizontal="right",  vertical="center")

# Border
thin           = Side(style="thin", color="CCCCCC")
border         = Border(left=thin, right=thin, top=thin, bottom=thin)

# Numeric columns to format
numeric_cols   = ["DTD_PL_US", "MTD_PL_US", "YTD_PL_US", "LTD_PL_US", "CASH", "CASH_US", "MV_US"]
num_format     = '#,##0.00'

# ── Format header row ──
for cell in ws[1]:
    cell.font      = header_font
    cell.fill      = header_fill
    cell.alignment = header_align
    cell.border    = border

ws.row_dimensions[1].height = 30

# ── Format body rows ──
col_indices = {cell.value: cell.column for cell in ws[1]}

for row in ws.iter_rows(min_row=2, max_row=ws.max_row):
    for cell in row:
        cell.font   = body_font
        cell.border = border
        col_name    = ws.cell(row=1, column=cell.column).value
        if col_name in numeric_cols:
            cell.number_format = num_format
            cell.alignment     = body_align_r
        else:
            cell.alignment     = body_align_l

    # Alternating row color
    if row[0].row % 2 == 0:
        for cell in row:
            cell.fill = PatternFill(fill_type="solid", fgColor="EEF2F7")

# ── Auto-fit column widths ──
for col in ws.columns:
    max_len = max((len(str(cell.value)) if cell.value else 0) for cell in col)
    ws.column_dimensions[get_column_letter(col[0].column)].width = min(max_len + 4, 40)

# ── Freeze header row ──
ws.freeze_panes = "A2"

# ── Add auto filter ──
ws.auto_filter.ref = ws.dimensions

wb.save(filepath)
print(f"✅ File saved : {filepath}")


✅ File saved : C:\Users\mkeita9\OneDrive - WBG\1. Work Documents\4. Projects & Initiatives\Murex_Testing_LAM_Cash\08_DR\DR4\01.Inputs\1_Datamart_Extracts\4_Murex_PL_Summary\MX_Subaccount_PL_20260814.xlsx


In [ ]:
# Cell 7 — Display the DataFrame
#display_df(df.head(10))   # first 10 rows
#display_df(df.tail(10))   # last 10 rows
display_df(df_filtered)            # everything

